# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget
# %matplotlib inline

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux
import skewerjax.utils.plotting_tools as pt
import skewerjax.utils.plotting_tools_3D_fields as pt3D_fields

import skewerjax.optical_depth.line_profile as lp
import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys

# Cosmology

In [ ]:
# Create cosmology object
cosmology = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time
ii_cosmology = 0

a_obs = 1.0

In [ ]:
L_box = 110.7  # Mpc
z_box = 2.0

N_grid_skewers = 3
N_ppsk = 4096
N_v = 4096

# Lines

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id = "HI-LyA")

# Fields

### Gaussian Random Field 

In [ ]:
T0_HI = 1.5*10**4
T0_Si = T0_HI

n0_min = 1e-16
n0_max = 1e-10

def pk_template(k: jnp.ndarray, A, n, k0, k_cut):
    # P(k) = A (k/k0)^n exp[-(k/k_cut)^2]
    kk = jnp.maximum(k, 1e-12)
    return A * (kk / k0) ** n * jnp.exp(-(kk / k_cut) ** 2)
number_density_sampler = toy_samplers.number_density_sampler_gaussian_3d(
    key=jax.random.PRNGKey(0),
    pk_of_k=lambda k: pk_template(k, A=5.0, n=-2.5, k0=2.0, k_cut=50.0),
    n0_min=n0_min, n0_max=n0_max, L_box=L_box,
    grid_shape=(256, 256, 256), contrast=1.2,
)
fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_HI, T0_HI, L_box)),    # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    }
})

# Skewers & optical depth

In [ ]:
# -------------------------------------------------------- #
# ---------------------- skewers ------------------------- #
# -------------------------------------------------------- #

points_grid_skewers = aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y")
points = jnp.reshape(points_grid_skewers, (N_grid_skewers**2, N_ppsk, 3))

skewer_bundle = SkewerBundle(spatial_points=points, cosmologies=cosmology, a0=1/(1+z_box), x0=L_box)

# -------------------------------------------------------- #
# ------------------- optical depth ---------------------- #
# -------------------------------------------------------- #

a_dagger = 1 / (1 + z_box)
a_em = skewer_bundle.scale_factors[0, 0, 0]
v_max = phys.c_kms * jnp.log(a_dagger / a_em)
velocities_kms = jnp.linspace(0, v_max, N_v)
dv = np.diff(velocities_kms)[0]

tau = od.OpticalDepthCalculator(
    skewer_bundle=skewer_bundle, velocities_kms=velocities_kms, line_bundle=line_bundle, fields=fields, a_dagger=a_dagger, debug_memory=True,
)
tau.compute()
optical_depth = tau.optical_depth

# -------------------------------------------------------- #
# --- Other spectral coordinates, utils, and wrapping ---- #
# -------------------------------------------------------- #
wavelengths_per_line_A = lp.velocities_to_line_centered_wavelengths(velocities_kms, line_bundle, a_dagger=a_dagger, a_obs=a_obs)  # (N_lines, N_v)
velocities_per_line_kms = lp.reference_velocities_to_line_centered(velocities_kms, line_bundle, reference_line_idx=0)  # (N_lines, N_v)

tau.wrap_to_ref_line()
optical_depth_wrapped = tau.optical_depth

v_min = -jnp.max(line_bundle.delta_v)

# -------------------------------------------------------- #
# ------------------------ Plots ------------------------- #
# -------------------------------------------------------- #
all_figures = pt3D_fields.generate_all_simulation_plots(
    line_bundle, skewer_bundle, fields, L_box, velocities_kms, wavelengths_per_line_A, velocities_per_line_kms,
    optical_depth, optical_depth_wrapped, a_dagger, a_em, a_obs, v_min, v_max, view_elev = 20.0, view_azim = -43.0,
)
# all_figures[0].savefig("hi_cylinder.png", dpi=300, bbox_inches='tight')

In [ ]:
# N_ppsk = 1024
# xx_LoS = jnp.linspace(start=0.0, stop=L_box, num=N_ppsk)
# scale_factors = cosmology.solve_a_of_x(
#     ICs=jnp.array([scale_factor_left]),
#     x_eval_Mpc=xx_LoS
# )[0]

# N_grid_skewers = 3
# points_grid_skewers = aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y")

# ic = N_grid_skewers//2
# jc = N_grid_skewers//2

# N_imshow = 1024
# points_grid_imshow = aux.make_plane_points(L_box, 'x', L_box/2, N_imshow, N_imshow)
# log10_number_density_imshow = np.log10(fields.number_density(jnp.concatenate([jnp.ones((N_imshow, N_imshow, 1)), points_grid_imshow], axis=-1)))

# density_norm = mpl.colors.Normalize(
#     vmin=float(np.min(log10_number_density_imshow)),
#     vmax=float(np.max(log10_number_density_imshow)),
# )

# skewer_cmap = mpl.cm.get_cmap("gist_rainbow")
# skewer_norm = mpl.colors.Normalize(vmin=0.0, vmax=float(N_ppsk - 1))

# list_plot = range(1, N_ppsk, 8) # plot one each 2 N_ppsk
# list_plot = [1, (N_ppsk-1)//2, N_ppsk-1]

# if len(list_plot) < 5:
#     show_plots = True
# else:
#     show_plots = False

# for ii in list_plot:

#     _N_ppsk = ii + 1
#     _points_grid_skewers = points_grid_skewers[:, :, 0:_N_ppsk, :]

#     _points = jnp.reshape(_points_grid_skewers, (N_grid_skewers*N_grid_skewers, _N_ppsk, 3))
    
#     a0_per_skewer = scale_factors[_N_ppsk-1]
#     x0_per_skewer = _points_grid_skewers[ic, jc, -1, 1]
#     skewer_bundle = SkewerBundle(spatial_points=_points, cosmologies=cosmology, a0=a0_per_skewer, x0=x0_per_skewer)
#     scale_factors_obs = jnp.repeat(scale_factor_obs, skewer_bundle.N_sk)
#     optical_depth = od.compute_optical_depth(skewer_bundle, wavelengths, line_bundle, fields, scale_factors_obs) # Compute optical depth
#     F_over_F0 = jnp.reshape(jnp.exp(-optical_depth), (line_bundle.n_lines, N_grid_skewers, N_grid_skewers, len(wavelengths)))

#     points_central = skewer_bundle.spatial_points[ic * N_grid_skewers + jc]
#     log10_number_density_central = np.log10(fields.number_density(jnp.concatenate([jnp.ones(points_central.shape[0:1])[..., None], points_central], axis=-1)))
#     F_over_F0_central = F_over_F0[..., ic * N_grid_skewers + jc]

#     fig1, axes1 = pt.plot_density_imshow_and_central_profile(
#         points_central=np.asarray(points_central),
#         points_grid_imshow=np.asarray(points_grid_imshow),
#         log10_number_density_imshow=np.asarray(log10_number_density_imshow),
#         log10_number_density_central=np.asarray(log10_number_density_central),
#         line_bundle=line_bundle, scale_factor_start=scale_factor_left, scale_factor_end=a0_per_skewer,
#         savepath_png=os.path.join(save_path_global, f"density_{ii}.png"),
#         density_norm=density_norm,
#         box_limits_yz=((0.0, L_box), (0.0, L_box)),
#         box_limits_los=(0.0, L_box),
#         skewer_cmap=skewer_cmap, skewer_norm=skewer_norm, show=show_plots,
#     )
    
#     fig2, axes2 = pt.plot_flux_panels(
#         wavelengths=wavelengths, F_over_F0=np.asarray(F_over_F0),
#         ic=ic, jc=jc, line_bundle=line_bundle,
#         scale_factor_start=scale_factor_left, scale_factor_end=a0_per_skewer,
#         x_axis_key=x_axis_key, min_x=min_plot, max_x=max_plot, lambda0_anchor=lambda0_anchor,
#         savepath_png=os.path.join(save_path_global, f"flux_split_{ii}.png"),
#         skewer_cmap=skewer_cmap, skewer_norm=skewer_norm,
#         skewer_last_index=ii,  # "last" point shown in this iteration
#         show=show_plots,
#     )

#     fig, ax = pt.plot_flux_panels_colored_by_los(
#         wavelengths=wavelengths, F_over_F0=np.asarray(F_over_F0),
#         ic=ic, jc=jc, line_bundle=line_bundle, x_axis_key=x_axis_key,
#         scale_factor_start=scale_factor_left, scale_factor_end=a0_per_skewer,
#         min_x=min_plot, max_x=max_plot, skewer_last_index=ii,
#         lambda0_anchor=lambda0_anchor, skewer_cmap=skewer_cmap, skewer_norm=skewer_norm,
#         savepath_png=os.path.join(save_path_global, f"flux_combined_{ii}.png"),
#         show=show_plots,
#     )

# Overflux

In [ ]:
overflux = sjx.optical_depth.overflux.DeltaFieldBuilder(optical_depth_wrapped).build()

# Summary stats

In [ ]:
delta_v_list = list(np.array(line_bundle.delta_v))
velocity_domain = np.max(tau.velocities_kms) - np.min(tau.velocities_kms)
stats = sjx.summary_stats.P1D_xi1D.OverfluxStatistics1D(overflux, dv)
ordered_pairs = stats.ordered_pairs

In [ ]:
stats.compute_p1d()

Pk_total_from_terms = stats.Pk_total_from_terms

Pk_total_per_sample = stats.Pk_total_per_sample
Pk_total = stats.Pk_total

Pk_pair_per_sample = stats.Pk_pair_per_sample
Pk_pair = stats.Pk_pair

k = stats.k

In [ ]:
fig, axes, axes_res, order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_by_order(
    ordered_pairs=ordered_pairs, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=velocity_domain, Npix=tau.N_v,
    tolerance_threshold=1e-6, use_log_x=True, # xlim=(2.0, 600),
    residual_mode="sum", show_harmonics=False, use_symlog_main=False, use_symlog_residual=False,
    symlog_linscale=1.0, symlog_label_minor=False, symlog_sci_cutoff=1e-2, legend_ncol_terms=2,
    xlabel=r"$k\,[{\rm s\,/km}]$", ylabel=r"$P_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_total_recovery(
    Pk_total=Pk_total, Pk_total_from_terms=Pk_total_from_terms, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=velocity_domain, Npix=tau.N_v,
    ordered_pairs=ordered_pairs, show_harmonics=False, show_order_sums=True, show_order_sum_residuals=True,
    use_symlog_main=True, use_symlog_residual=False, xlabel=r"$k\,[{\rm s\,/km}]$"
)

In [ ]:
# ===============================================================
# Build order sums from Pk_pair

ordered_pairs = list(Pk_pair.keys()) if ordered_pairs is None else ordered_pairs
order_sum_dict = {}
for pair in ordered_pairs:
    A, B = pair
    order = len(A) + len(B)
    if order not in order_sum_dict:
        order_sum_dict[order] = np.zeros_like(Pk_pair[pair], dtype=float)
    order_sum_dict[order] += Pk_pair[pair].real
orders = sorted(order_sum_dict.keys())

# ===============================================================
# Plot P1D

cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(12, 9))
for i, order in enumerate(orders):
    y_order = order_sum_dict[order][1:] * k[1:] / np.pi
    ax.plot(k[1:], y_order, lw=2.0, color=cmap(i % 10), alpha=0.9, label=rf"$\mathcal{{O}}({order})$")

# Total
ax.plot(k[1:], Pk_total.real[1:] * k[1:] / np.pi, lw=3.0, color="k", label=r"Tot.")

# Total recovered from terms
ax.plot(k[1:], Pk_total_from_terms.real[1:] * k[1:] / np.pi, lw=2.0, color="red", ls="--", label=r"Tot. Sum")

# ax.set_yscale("log")

ax.set_xlabel(r"$k\,[{\rm s\,/km}]$", fontsize=22)
ax.set_ylabel(r"$k P_{\rm 1D} / \pi$", fontsize=22)

ax.set_xlim([0.0, 0.1])
ax.set_ylim([1e-4, 1e0])

ax.tick_params(axis="both", labelsize=20)
ax.legend(fontsize=20, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
stats.compute_xi1d()

xi_total_from_terms = stats.xi_total_from_terms

xi_total_per_sample = stats.xi_total_per_sample
xi_total = stats.xi_total

xi_pair_per_sample = stats.xi_pair_per_sample
xi_pair = stats.xi_pair

lags = stats.lags

In [ ]:
fig, axes, axes_res, xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_by_order(
    ordered_pairs=ordered_pairs, xi_pair=xi_pair, lags=lags, delta_v_list=delta_v_list, L_box=velocity_domain,
    tolerance_threshold=1e-6, residual_mode="sum", show_delta_v_diff_lines=True, use_symlog_main=False,
    use_symlog_residual=False, symlog_linscale=1.0, symlog_base=10, symlog_label_minor=False, symlog_sci_cutoff=1e-2,
    show_order_badge=True, legend_ncol_terms=2, xlabel=r"$v\,\left[{\rm km / s}\right]$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
    xlim=(-0.0, 3000), # xlim=(-0.0, v_max/2)
)

fig, (ax, ax_res), xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_total_recovery(
    xi_total=xi_total, xi_total_from_terms=xi_total_from_terms, xi_pair=xi_pair, lags=lags, delta_v_list=delta_v_list, L_box=velocity_domain, ordered_pairs=ordered_pairs,
    show_order_sums=True, show_order_sum_residuals=True, show_delta_v_diff_lines=True, annotate_half_line=False,
    half_line_text=r"$L_\mathrm{box}/2$", use_symlog_main=False, use_symlog_residual=True, main_symlog_linthresh=None,
    symlog_linscale=1.0, symlog_base=10, symlog_sci_cutoff=1e-2, xlabel=r"$v\,\left[{\rm km / s}\right]$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
    xlim=(-0.0, 3000), # xlim=(-0.0, v_max/2)
)